In [ ]:
import shutil
import yaml
from pathlib import Path
from typing import List, Tuple, Dict, Any
from PIL import Image
from tqdm import tqdm
import torch
import numpy as np
from skimage import measure
from ultralytics.models.sam import SAM3Predictor

In [ ]:
def create_predictor(model_path: str = "/kaggle/input/datasets/roadwarrior23513245r/samyolo/SAM.pt", imgsz: int = 644) -> SAM3Predictor:
    """инициализация SAM """
    return SAM3Predictor(
        overrides=dict(
            conf=0.25, task="segment", mode="predict", model=model_path,
            save=False, imgsz=imgsz, verbose=False
        )
    )

def read_yolo_labels(label_path: Path, img_width: int, img_height: int) -> Tuple[np.ndarray, List[int]]:
    """чтение YOLO bbox в пиксели + class_id"""
    boxes, class_ids = [], []
    try:
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = int(parts[0])
                    x_center, y_center, width, height = map(float, parts[1:5])
                    x1, y1 = (x_center - width/2) * img_width, (y_center - height/2) * img_height
                    x2, y2 = (x_center + width/2) * img_width, (y_center + height/2) * img_height
                    boxes.append([x1, y1, x2, y2])
                    class_ids.append(class_id)
    except FileNotFoundError:
        pass
    return np.array(boxes), class_ids


def masks_to_yolo_polygons(results, class_ids: List[int], img_width: int, img_height: int) -> List[List[float]]:
    """конверитация SAM масок в YOLO полигоны"""
    polygons = []
    mask_idx = 0
    
    for r in results:
        if r.masks is not None:
            masks = r.masks.data.cpu().numpy()
            for mask in masks:
                if mask_idx < len(class_ids):
                    contours = measure.find_contours(mask, 0.5)
                    if contours:
                        contour = max(contours, key=len)
                        polygon = contour[:, [1, 0]]  # x,y
                        polygon[:, 0] /= img_width
                        polygon[:, 1] /= img_height
                        polygon = np.round(polygon, 6)
                        
                        if 3 <= len(polygon) <= 2000:
                            polygons.append([class_ids[mask_idx]] + polygon.flatten().tolist())
                        mask_idx += 1
    
    return polygons


def save_yolo_labels(polygons: List[List[float]], label_path: Path) -> bool:
    """сохранение YOLO segmentation labels"""
    if not polygons:
        return False
    with open(label_path, 'w') as f:
        for polygon in polygons:
            f.write(" ".join(map(str, polygon)) + "\n")
    return True


def process_dataset_split(predictor: SAM3Predictor, img_dir: Path, label_dir: Path, 
                         output_dir: Path, class_names: List[str], desc: str) -> int:
    """обработка одного сплита датасета"""
    img_paths = sorted(img_dir.glob("*.[jJ][pP][gG]*")) + sorted(img_dir.glob("*.[pP][nN][gG]*"))
    img_paths = [p for p in img_paths if (label_dir / f"{p.stem}.txt").exists()]
    
    count = 0
    print(f" {desc} ({len(img_paths)} изображений)...")
    
    for img_path in tqdm(img_paths, desc=desc, ncols=100):
        with Image.open(img_path) as img:
            w, h = img.size
        
        label_path = label_dir / f"{img_path.stem}.txt"
        boxes, class_ids = read_yolo_labels(label_path, w, h)
        
        if len(boxes) > 0:
            predictor.set_image(str(img_path))
            results = predictor(bboxes=boxes)
            
            polygons = masks_to_yolo_polygons(results, class_ids, w, h)
            out_label = output_dir / "labels" / f"{img_path.stem}.txt"
            
            if save_yolo_labels(polygons, out_label):
                shutil.copy(img_path, output_dir / "images" / img_path.name)
                count += 1
            
            # очистка памяти
            del results
            predictor.reset_image()
            torch.cuda.empty_cache()
    
    return count


def from_det_to_seg_func():
    """Главная функция: bbox -> segmentation для всех сплитов"""
    dataset_root = Path("/kaggle/working/yolo_seg_dataset_WaRP") # путь к датасету
    
    # создание структуры
    for split in ["train", "val", "test"]:
        for typ in ["images", "labels"]:
            (dataset_root / split / typ).mkdir(parents=True, exist_ok=True)
    
    # пути к данным
    data_paths = {
        "train": (Path("/kaggle/input/datasets/parohod/warp-waste-recycling-plant-dataset/Warp-D/train/images"),
                  Path("/kaggle/input/datasets/parohod/warp-waste-recycling-plant-dataset/Warp-D/train/labels")),
        "val": (Path("/kaggle/input/datasets/parohod/warp-waste-recycling-plant-dataset/Warp-D/val/images"),
                Path("/kaggle/input/datasets/parohod/warp-waste-recycling-plant-dataset/Warp-D/val/labels")),
        "test": (Path("/kaggle/input/datasets/parohod/warp-waste-recycling-plant-dataset/Warp-D/test/images"),
                 Path("/kaggle/input/datasets/parohod/warp-waste-recycling-plant-dataset/Warp-D/test/labels"))
    }
    
    predictor = create_predictor()
    class_names = ['bottle-blue', 'bottle-green', 'bottle-dark', 'bottle-milk',
            'bottle-transp', 'bottle-multicolor', 'bottle-yogurt', 'bottle-oil',
            'cans', 'juice-cardboard', 'milk-cardboard', 'detergent-color',
            'detergent-transparent', 'detergent-box', 'canister',
            'bottle-blue-full', 'bottle-transp-full', 'bottle-dark-full',
            'bottle-green-full', 'bottle-multicolor-full', 'bottle-milk-full',
            'bottle-oil-full', 'detergent-white', 'bottle-blue5l',
            'bottle-blue5l-full', 'glass-transp', 'glass-dark', 'glass-green']
    
    stats = {}
    for split, (img_dir, lbl_dir) in data_paths.items():
        stats[split] = process_dataset_split(
            predictor, img_dir, lbl_dir, dataset_root / split, class_names, split.upper()
        )
    
    # data.yaml
    data_yaml = {
        "path": str(dataset_root.absolute()),
        "train": "train/images", "val": "val/images", "test": "test/images",
        "names": {i: name for i, name in enumerate(class_names)}
    }
    with open(dataset_root / "data.yaml", "w") as f:
        yaml.dump(data_yaml, f, default_flow_style=False)
    
    print(f"\nTrain: {stats['train']}, Val: {stats['val']}, Test: {stats['test']}")
    print(f"dataset_root: {dataset_root}")



In [ ]:
from_det_to_seg_func()